# 02 — stdio Transport: Sending Messages Over Pipes

**Goal:** Build the transport layer — how JSON-RPC messages travel between client and server.

## How MCP stdio transport works

```
Client process                    Server process
┌──────────┐                      ┌──────────┐
│          │ ── client writes ──→ │ stdin    │
│          │                      │          │
│          │ ←── server writes ── │ stdout   │
│          │                      │          │
│          │    (logs only) ←──── │ stderr   │
└──────────┘                      └──────────┘
```

**Rules:**
1. Each message = one JSON object on one line (newline-delimited)
2. Server reads from stdin, writes to stdout
3. Server MUST NOT print anything else to stdout (use stderr for logs)
4. Client launches server as a subprocess

## Exercise 2.1 — Write a message writer and reader

These functions convert between Python dicts and newline-delimited JSON strings.

In [ ]:
import json
import io

def write_message(stream, message: dict):
    """Write a JSON-RPC message to a stream (one line, newline-terminated)."""
    # TODO: json.dumps the message (no extra whitespace), add newline, flush
    # Hint: use ensure_ascii=False for unicode support
    pass

def read_message(stream) -> dict | None:
    """Read one JSON-RPC message from a stream. Returns None on EOF."""
    # TODO: read one line, strip whitespace, parse as JSON
    # Return None if line is empty (EOF)
    pass


# Test with in-memory streams (simulating stdin/stdout)
buffer = io.StringIO()

# Write messages
write_message(buffer, {"jsonrpc": "2.0", "id": 1, "method": "tools/list"})
write_message(buffer, {"jsonrpc": "2.0", "id": 1, "result": {"tools": []}})

# Read them back
buffer.seek(0)
msg1 = read_message(buffer)
msg2 = read_message(buffer)
msg3 = read_message(buffer)  # should be None (EOF)

print("Message 1:", msg1)
print("Message 2:", msg2)
print("Message 3 (EOF):", msg3)

assert msg1["method"] == "tools/list"
assert msg2["result"]["tools"] == []
assert msg3 is None

print("\n✓ All tests passed!")

## Exercise 2.2 — Write a standalone server script

Let's write a minimal server that runs as a separate process. It reads from stdin, processes requests, writes to stdout.

Run the next cell to create the file, then we'll test it.

In [ ]:
server_code = '''\
#!/usr/bin/env python3
"""Minimal JSON-RPC server over stdio.
Reads JSON lines from stdin, writes JSON lines to stdout.
"""
import json
import sys

def log(msg):
    """Log to stderr (NEVER stdout — stdout is for JSON-RPC only)."""
    print(f"[server] {msg}", file=sys.stderr, flush=True)

def handle_add(params):
    return params["a"] + params["b"]

def handle_echo(params):
    return params.get("text", "")

HANDLERS = {
    "add": handle_add,
    "echo": handle_echo,
}

def process_message(raw_line: str) -> str | None:
    """Process one JSON-RPC message. Returns response JSON string, or None for notifications."""
    msg = json.loads(raw_line)
    method = msg.get("method", "")
    msg_id = msg.get("id")  # None for notifications
    params = msg.get("params", {})
    
    # Notification — no response
    if msg_id is None:
        log(f"notification: {method}")
        return None
    
    # Request — must respond
    handler = HANDLERS.get(method)
    if handler is None:
        return json.dumps({"jsonrpc": "2.0", "id": msg_id, "error": {"code": -32601, "message": f"Unknown method: {method}"}})
    
    try:
        result = handler(params)
        return json.dumps({"jsonrpc": "2.0", "id": msg_id, "result": result})
    except Exception as e:
        return json.dumps({"jsonrpc": "2.0", "id": msg_id, "error": {"code": -32603, "message": str(e)}})

def main():
    log("started, waiting for input...")
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        log(f"received: {line[:80]}")
        response = process_message(line)
        if response:
            print(response, flush=True)  # to stdout
            log(f"sent: {response[:80]}")
    log("stdin closed, shutting down")

if __name__ == "__main__":
    main()
'''

with open("simple_rpc_server.py", "w") as f:
    f.write(server_code)
print("Created simple_rpc_server.py")

## Exercise 2.3 — Talk to the server via subprocess

Now write a client that launches the server as a subprocess and communicates via stdin/stdout pipes.

In [ ]:
import subprocess
import json

def call_server(requests: list[dict]) -> list[dict]:
    """Launch server subprocess, send requests, collect responses."""
    # TODO:
    # 1. Build the input: one JSON line per request, joined by newlines
    # 2. Launch subprocess with stdin=PIPE, stdout=PIPE, stderr=PIPE
    #    Command: ["python3", "simple_rpc_server.py"]
    # 3. Send all input via proc.communicate(input=...)
    # 4. Parse each stdout line as a JSON response
    # 5. Return list of response dicts
    pass


# Test it
responses = call_server([
    {"jsonrpc": "2.0", "id": 1, "method": "add", "params": {"a": 10, "b": 20}},
    {"jsonrpc": "2.0", "id": 2, "method": "echo", "params": {"text": "hello MCP!"}},
    {"jsonrpc": "2.0", "method": "some_notification"},  # no id → no response
    {"jsonrpc": "2.0", "id": 3, "method": "unknown_method"},
])

for resp in responses:
    print(json.dumps(resp, indent=2))
    print()

# Should get 3 responses (notification doesn't get one)
assert len(responses) == 3
assert responses[0]["result"] == 30        # add
assert responses[1]["result"] == "hello MCP!"  # echo
assert responses[2]["error"]["code"] == -32601  # unknown method

print("✓ All tests passed!")

### Question 2.3
We sent 4 messages but got only 3 responses. Why? What happened to the notification?

*Your answer:*


## Exercise 2.4 — Async version (interactive, long-lived)

The subprocess approach above sends all messages at once and waits for the process to exit. But MCP servers are **long-lived** — the client sends messages interactively over time.

Let's use `asyncio.create_subprocess_exec` to keep the server running and send messages one at a time.

In [ ]:
import asyncio
import json

async def interactive_session():
    """Launch server, send requests one at a time, read responses."""
    proc = await asyncio.create_subprocess_exec(
        "python3", "simple_rpc_server.py",
        stdin=asyncio.subprocess.PIPE,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
    )
    
    async def send(msg: dict) -> dict | None:
        """Send a JSON-RPC message and read the response (if expected)."""
        line = json.dumps(msg) + "\n"
        proc.stdin.write(line.encode())
        await proc.stdin.drain()
        
        # Only read response if it's a request (has id)
        if "id" in msg:
            resp_line = await proc.stdout.readline()
            return json.loads(resp_line.decode())
        return None
    
    # Interactive conversation with the server
    print("--- Session start ---")
    
    r1 = await send({"jsonrpc": "2.0", "id": 1, "method": "add", "params": {"a": 5, "b": 3}})
    print(f"5 + 3 = {r1['result']}")
    
    r2 = await send({"jsonrpc": "2.0", "id": 2, "method": "echo", "params": {"text": "live session!"}})
    print(f"Echo: {r2['result']}")
    
    # Send a notification (no response)
    await send({"jsonrpc": "2.0", "method": "ping"})
    print("Sent notification (no response expected)")
    
    r3 = await send({"jsonrpc": "2.0", "id": 3, "method": "add", "params": {"a": 100, "b": 200}})
    print(f"100 + 200 = {r3['result']}")
    
    # Shutdown: close stdin → server exits
    proc.stdin.close()
    await proc.wait()
    print("--- Session end ---")

await interactive_session()

### Question 2.4
How does the server know when to shut down? What happens when we call `proc.stdin.close()`?

*Your answer:*


## Summary

You now have a working transport layer:

| Component | What it does |
|-----------|--------------|
| `write_message()` | Serialize dict → JSON line to stream |
| `read_message()` | Read JSON line from stream → dict |
| `simple_rpc_server.py` | Standalone server process (reads stdin, writes stdout) |
| `interactive_session()` | Async client that talks to server via subprocess pipes |

**Key insight:** The transport is dumb — it just moves JSON lines between processes. All the intelligence (methods, tools, handshaking) lives in the protocol layer above.

**Next notebook:** Add the MCP protocol on top — initialization handshake, tool discovery, tool calling